# stack-vs-cat — worked example 3: Interleave two vectors via stack + reshape

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `stack-vs-cat`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

To interleave `a` and `b` into `[a0, b0, a1, b1, ...]`, stack them along a new last axis to get `(n, 2)` pairs, then flatten row-major. Because the row-major walk visits the inner (pair) axis fastest, each `(a_i, b_i)` pair is emitted together before advancing. This composes stack (rank up) with reshape (collapse).

## Worked solution

Given equal-length 1-D `a` and `b`, `t.stack([a, b], dim=1)` produces a `(n, 2)` matrix where row i is `[a_i, b_i]`. Flattening with `.reshape(-1)` walks row-major: position 0,0 then 0,1 then 1,0 ... which is exactly `a0, b0, a1, b1, ...`. We print the interleaved 1-D result of length `2n` to confirm the alternating pattern. The pipeline is rank-up-then-flatten, not a single op.

In [ ]:
a = t.tensor([1, 3, 5, 7])
b = t.tensor([2, 4, 6, 8])
interleaved = t.stack([a, b], dim=1).reshape(-1)
print('pairs shape:', tuple(t.stack([a, b], dim=1).shape))
print('interleaved:', interleaved.tolist())